<a href="https://colab.research.google.com/github/carlogr-13/Dynamic_Allostery_PLMs/blob/main/perturbation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.2. Perfil perturbación epistática y propagación distal

Magnitud del recableado inducido por la mutación I150A mediante análisis diferencial de tensores de acoplamiento, evaluando tanto el impacto directo sobre el andamiaje hidrofóbico como la transmisión de señal epistática a larga distancia.

*   Matriz maestra de impacto diferencial
*   Mapeo de propagación distal



## 4.2.1. Impacto directo en el andamiaje hidrofóbico basal

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. MONTAJE Y NAVEGACIÓN A LA CARPETA DE TOPOLOGÍA
drive.mount('/content/drive')
ruta_topologia = '/content/drive/MyDrive/TFG_notebooks/Data_PKA/quantitative_metrics/'
os.chdir(ruta_topologia)

target_residues = [57, 70, 95, 106, 128, 164, 172, 173, 174, 185, 220, 222, 227, 231]
master_df = None

print("Construyendo la Matriz Maestra de Perturbación (Versión Corregida)...")

for res in target_residues:
    wt_file = f"DirectedSensitivity_{res}_WT.csv"
    mut_file = f"DirectedSensitivity_{res}_I150A.csv"

    if os.path.exists(wt_file) and os.path.exists(mut_file):
        df_wt = pd.read_csv(wt_file)
        df_mut = pd.read_csv(mut_file)

        # EL CAMBIO CLAVE: Fusionamos SOLO por la columna numérica 'Residue_PDB'
        df_merged = pd.merge(df_wt, df_mut, on="Residue_PDB", suffixes=('_WT', '_Mut'))

        # Calculamos el Delta (Mutante - WT)
        df_merged[f'Delta_JSD_{res}'] = df_merged['Epistatic_Coupling_JSD_Mut'] - df_merged['Epistatic_Coupling_JSD_WT']

        # Nos quedamos con el número, la letra original (WT) y el Delta calculado
        df_delta = df_merged[['Residue_PDB', 'Amino_Acid_WT', f'Delta_JSD_{res}']]

        # Renombramos la columna para que quede limpia en la tabla final
        df_delta = df_delta.rename(columns={'Amino_Acid_WT': 'Amino_Acid'})

        # Fusión en la tabla maestra
        if master_df is None:
            master_df = df_delta
        else:
            # Ahora la fusión maestra también funciona perfectamente sin perder datos
            master_df = pd.merge(master_df, df_delta, on=['Residue_PDB', 'Amino_Acid'], how='outer')
    else:
        print(f"  [!] Faltan archivos para el residuo {res}")

if master_df is not None:
    # Limpiamos huecos vacíos
    master_df = master_df.fillna(0)

    # Suma Global de Impacto Absoluto
    delta_cols = [col for col in master_df.columns if 'Delta_JSD' in col]
    master_df['Impacto_Global_Absoluto'] = master_df[delta_cols].abs().sum(axis=1)

    # Ordenamos de mayor a menor perturbación
    master_df = master_df.sort_values(by='Impacto_Global_Absoluto', ascending=False)

    # Exportamos el CSV definitivo
    master_df.to_csv("Master_Delta_JSD_Matrix_Completa.csv", index=False)

    print("\n¡ÉXITO! Matriz corregida generada con éxito.")
    print("\nAquí tienes el Top verdadero de residuos que más perturban la red:")
    print(master_df[['Residue_PDB', 'Amino_Acid', 'Impacto_Global_Absoluto']].head(5))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Construyendo la Matriz Maestra de Perturbación (Versión Corregida)...

¡ÉXITO! Matriz corregida generada con éxito.

Aquí tienes el Top verdadero de residuos que más perturban la red:
     Residue_PDB Amino_Acid  Impacto_Global_Absoluto
135          150          I                 0.020660
98           113          N                 0.008640
220          235          Y                 0.006968
138          153          T                 0.005632
127          142          H                 0.004680


## 4.2.2. Propagación distal del recableado alostérico

In [ ]:
# 1. Definimos los archivos
wt_file = "LongRange_Allostery_WT.csv"
mut_file = "LongRange_Allostery_I150A.csv"

print("Iniciando escrutinio de propagación alostérica de largo alcance (>15 Å)...")

if os.path.exists(wt_file) and os.path.exists(mut_file):
    df_wt = pd.read_csv(wt_file)
    df_mut = pd.read_csv(mut_file)

    # Adaptabilidad a los nombres de las columnas de tu script
    col_source = "Source_PDB" if "Source_PDB" in df_wt.columns else df_wt.columns[0]
    col_target = "Target_PDB" if "Target_PDB" in df_wt.columns else df_wt.columns[2]
    col_weight = "Weight" if "Weight" in df_wt.columns else "Epistatic_Coupling_JSD"

    # 2. Fusión de los grafos espaciales (alineamos por origen y destino)
    df_merged = pd.merge(df_wt, df_mut, on=[col_source, col_target], suffixes=('_WT', '_Mut'))

    # 3. Filtro Diferencial: Calculamos el Delta JSD (Mutante - WT)
    df_merged['Delta_JSD'] = df_merged[f'{col_weight}_Mut'] - df_merged[f'{col_weight}_WT']

    # 4. Filtro de Consolidación: Solo queremos rutas que AUMENTAN su transferencia
    # y las ordenamos de mayor a menor incremento.
    df_sorted = df_merged[df_merged['Delta_JSD'] > 0].sort_values(by='Delta_JSD', ascending=False)

    print("\n" + "="*70)
    print("TOP 5: RUTAS ALOSTÉRICAS EMERGENTES O FORTALECIDAS")
    print("="*70)

    # Mostrar resultados limpios
    for index, row in df_sorted.head(5).iterrows():
        source = int(row[col_source])
        target = int(row[col_target])

        # Buscar si la distancia se mantuvo en una columna específica tras el merge
        dist_col = 'Distance_WT' if 'Distance_WT' in df_merged.columns else ('Distance_Angstroms_WT' if 'Distance_Angstroms_WT' in df_merged.columns else [c for c in df_merged.columns if 'Dist' in c][0])
        dist = row[dist_col]

        wt_val = row[f'{col_weight}_WT']
        mut_val = row[f'{col_weight}_Mut']
        delta = row['Delta_JSD']

        print(f"Ruta: {source} ---> {target} | Distancia: {dist:.2f} Å | Delta: +{delta:.5f} (WT: {wt_val:.5f} -> Mut: {mut_val:.5f})")

else:
    print("\n[!] ERROR: No se encuentran los archivos LongRange_Allostery_WT.csv y LongRange_Allostery_I150A.csv")

Iniciando escrutinio de propagación alostérica de largo alcance (>15 Å)...

TOP 5: RUTAS ALOSTÉRICAS EMERGENTES O FORTALECIDAS
Ruta: 142 ---> 157 | Distancia: 23.18 Å | Delta: +0.00380 (WT: 0.01553 -> Mut: 0.01933)
Ruta: 142 ---> 156 | Distancia: 22.00 Å | Delta: +0.00368 (WT: 0.00250 -> Mut: 0.00618)
Ruta: 106 ---> 113 | Distancia: 21.40 Å | Delta: +0.00313 (WT: 0.01320 -> Mut: 0.01634)
Ruta: 45 ---> 52 | Distancia: 18.29 Å | Delta: +0.00202 (WT: 0.01372 -> Mut: 0.01574)
Ruta: 190 ---> 205 | Distancia: 18.95 Å | Delta: +0.00175 (WT: 0.00831 -> Mut: 0.01006)
